Step 1: Install Dependencies

In [2]:
# Install required libraries for RAG system

!pip install -q langchain==0.3.25 langchain-community langchain-groq chromadb fastembed pypdf

# langchain        → main framework for building LLM apps (RAG, agents)
# langchain-core   → core components (prompts, messages, chains)
# langchain-community → extra tools (PDF loader, vector DB helpers)
# langchain-groq    → connects Groq LLM API (fast LLM inference)
# chromadb          → vector database for storing embeddings
# fastembed         → converts text into embeddings (vectors)
# pypdf             → reads and extracts text from PDF files

step 2: Import Libraries

In [3]:
# Import required libraries for building RAG system

import os  # for setting environment variables (API keys)

from langchain_community.document_loaders import PyPDFLoader
# loads PDF file and converts it into readable text documents

from langchain.text_splitter import RecursiveCharacterTextSplitter
# splits large text into small chunks for better retrieval

from langchain_community.embeddings import FastEmbedEmbeddings
# converts text into numerical vectors (embeddings)

from langchain_community.vectorstores import Chroma
# stores embeddings and allows similarity search

from langchain_groq import ChatGroq
# connects to Groq LLM for fast text generation

from langchain_core.prompts import ChatPromptTemplate
# creates structured prompts for LLM

from langchain.chains import create_retrieval_chain
# connects retriever + LLM into full RAG pipeline

from langchain.chains.combine_documents import create_stuff_documents_chain
# combines retrieved documents into one context for LLM

Step 3: Set API Key

In [ ]:
# Set Groq API key for accessing LLM

os.environ["GROQ_API_KEY"] = "your key here"

# This allows LangChain to connect with Groq cloud models
# without this, LLM will not work

STEP 4: Load PDF Document

In [17]:
pdf_path = "/content/UNIT V - Machine Learning AI NOTE - KBS (Draft 1).pdf"
loader = PyPDFLoader(pdf_path) #Load PDF using LangChain PDF loader
pages = loader.load() #Extract all pages from the PDF
print(f"Loaded {len(pages)} pages.")
print(f"Page 1 preview: {pages[0].page_content[:300]}...")


Loaded 14 pages.
Page 1 preview: MACHINE LEARNING  
 
1  Krishna Bikram Shah, nec 
 
 
LEARNING    
Learning process is the basis of knowledge acquisition process. Knowledge acquisition is the expanding the 
capabilities of a system or improving its performance at some specified task. So we can say knowledge 
acquisition is the goa...


Step 5 : Chunking (Split pages into smaller parts)

In [6]:
# Split PDF pages into smaller text chunks for better search accuracy in RAG

from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   # each chunk size (approx characters)
    chunk_overlap=50  # overlap helps preserve context between chunks
)
# Convert pages into chunks
chunks = splitter.split_documents(pages)

# Show how many chunks were created
print(f"Total chunks created: {len(chunks)}")

# Preview first chunk
print(chunks[0].page_content[:300])

Total chunks created: 78
MACHINE LEARNING  
 
1  Krishna Bikram Shah, nec 
 
 
LEARNING    
Learning process is the basis of knowledge acquisition process. Knowledge acquisition is the expanding the 
capabilities of a system or improving its performance at some specified task. So we can say knowledge 
acquisition is the goa


Step 6: Embeddings (Text → Vectors) + build vector store

In [8]:
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

# Free fast embedding model
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Create vector database
vectorstore = Chroma.from_documents(
    chunks,
    embeddings
)

print("Vector store ready!")

Vector store ready!


Step 8: Create Retriever (Search Engine)

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

results = retriever.invoke("What is this book about?")
for i, doc in enumerate(results):
    print(f" Result {i+1} (page {doc.metadata.get('page', '?')})")
    print(doc.page_content[:200])
    print()

 Result 1 (page 5)
between the objects and the actions.     
Explanation based generalization (EBG) is an algorithm for explanation based learning, described in Mitchell 
at al. (1986). It has two steps first, explain m

 Result 2 (page 0)
MACHINE LEARNING  
 
1  Krishna Bikram Shah, nec 
 
 
LEARNING    
Learning process is the basis of knowledge acquisition process. Knowledge acquisition is the expanding the 
capabilities of a system 

 Result 3 (page 4)
Learning from Advices    
In this process we can learn through taking advice from others. The idea of advice taking learning was 
proposed in early 1958 by McCarthy. In our daily life, this learning p

 Result 4 (page 3)
system to select and transform knowledge into a usable form and then integrate it into the existing knowledge 
of the system. It is a more complex  form of learning. This learning technique requires m

 Result 5 (page 4)
The computer follows the instructions given by the programmer. Hence, a kind of learning takes pl

Step 10 : Build RAG Chain (LLM + Retriever + Answer)

In [12]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Load Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Prompt template for RAG
# prompt = ChatPromptTemplate.from_template("""
# Answer the question based ONLY on the following context.
# If the context doesn't contain the answer, say "I don't know".

# Context: {context}
# Question: {input}
# """)
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Use the context to answer the question.

If context is partial, combine ideas and summarize logically.

Context:
{context}

Question:
{input}
""")

# Combine LLM + prompt
doc_chain = create_stuff_documents_chain(llm, prompt)

# Final RAG chain (uses your existing retriever)
rag_chain = create_retrieval_chain(retriever, doc_chain)

print("RAG chain ready!")


RAG chain ready!


Step 11: Ask Questions !

In [13]:
query = "What is the main message of this book?"

response = rag_chain.invoke({"input": query})

print("Answer:")
print(response["answer"])

print("\n--- Source chunks used ---")

docs = response.get("context", response.get("context_documents", []))

for i, doc in enumerate(docs):
    print(f"\nSource {i+1}: Page {doc.metadata.get('page', '?')}")
    print(doc.page_content[:200])

Answer:
Based on the provided context, it appears that the main message of this book revolves around different types of learning and communication among agents. The book discusses various learning techniques, including:

1. Communication Learning: Emphasizing the importance of understanding each agent's knowledge, reliability, and capability to improve performance through efficient communication.
2. Group Observation and Discovery Learning: Highlighting the effectiveness of combining different information and knowledge to learn new concepts.
3. Explanation-Based Generalization (EBG): Describing an algorithm for explanation-based learning, which involves pruning away unimportant aspects and generalizing explanations.
4. Learning from Advices: Discussing the process of learning through taking advice from others, which is a common learning process in daily life.
5. Learning by Deduction: Explaining the process of deriving new facts or relationships through deductive inference steps using 

In [14]:
query = "What are the different types of learning described in the book?"

response = rag_chain.invoke({"input": query})

print(response["answer"])

Based on the provided context, the different types of learning described in the book are:

1. **Learning by Acquisition**: This type of learning involves acquiring knowledge through knowledge acquisition techniques. The acquired knowledge may be required for a same problem in the future, making it easier to solve.

2. **Learning by Deduction**: This process involves a sequence of deductive inference steps using known facts to derive new facts or relationships. It requires more inference than rote learning and includes learning from teachers and books.

3. **Learning by Generalization**: This type of learning involves generalization from experience, where a computer system is able to perform similar tasks in a related domain more effectively.

4. **Skill Refinement**: This refers to the improvement of skills by performing the same task again and again.

5. **Inductive Learning**: This form of learning involves formulating a general concept after seeing a number of instances. It is more 

In [15]:
query = "Explain explanation-based generalization (EBG) in simple terms"

response = rag_chain.invoke({"input": query})

print(response["answer"])

Explaination-based generalization (EBG) is a type of machine learning algorithm that helps a computer learn from examples and create a general rule or concept. Here's how it works in simple terms:

1. **Understanding the Goal**: The computer is given a specific goal or concept to learn, like "what is a bucket?"
2. **Analyzing the Example**: The computer is shown a single example of a bucket, like a picture or a description.
3. **Identifying Key Features**: The computer uses this example to identify the key features of a bucket, like its shape, size, and material.
4. **Generalizing the Rule**: The computer then uses these key features to create a general rule or concept of what a bucket is, like "a bucket is a container with a round shape and a handle."
5. **Applying the Rule**: The computer can then use this general rule to identify other examples of buckets and make predictions about new, unseen examples.

EBG is useful because it helps computers learn from a small number of examples 

In [16]:
query = "What is the difference between inductive learning and deductive learning?"

response = rag_chain.invoke({"input": query})

print(response["answer"])

Based on the provided context, the main difference between inductive learning and deductive learning lies in their approach to deriving new knowledge or facts.

**Inductive Learning:**
Inductive learning involves trying to induce a general rule based on observed instances. It's a process of making an educated guess or forming a hypothesis based on specific inputs and outputs. The system tries to infer an association between specific inputs and outputs, and it's often used to classify subsequent instances. For example, if we observe that a certain taste is sweet in multiple instances, we can induce that the taste of sugar is sweet.

**Deductive Learning:**
Deductive learning, on the other hand, involves deriving new facts or relationships from known facts through a sequence of deductive inference steps. It's a process of logically deriving new information from existing knowledge. For example, if we know that x is the cousin of y, and we have the knowledge of x's and y's parents and the 

CONCLUSION

In this project, we successfully built a Retrieval-Augmented Generation (RAG) system using LangChain, Chroma vector database, and FastEmbed embeddings.

The PDF documents were processed and split into meaningful chunks using RecursiveCharacterTextSplitter, which improved the quality of retrieval. These chunks were then converted into vector embeddings using the BAAI/bge-small-en-v1.5 model and stored in Chroma for efficient semantic search.

A Groq LLM (Llama-3.1-8B-Instant) was integrated to generate responses based on retrieved context. The system was able to answer questions, explain concepts, and summarize information effectively.

Performance improved significantly after:
- Allowing the model to infer and summarize from partial context

Overall, the RAG system is now capable of producing context-aware and meaningful answers from uploaded documents, demonstrating a complete and functional end-to-end pipeline.